# MasterMind AI Standalone Review Workbook

This notebook is intentionally **independent**.

What that means:
- It does **not** import your project files from `src/`
- It does **not** depend on `requirements.txt`
- It does **not** require your Flask app, training code, or feature engineering code
- It uses only Python standard-library modules

Why this version is useful:
- You can open it in almost any Python/Jupyter setup
- You can explain your project clearly to a guide or evaluator
- You can use it as a structured project review notebook
- You can demonstrate the **logic** of your credit scoring workflow without worrying about environment issues

Important note:
This notebook is a **review and demo workbook**, not a replacement for your production code. Your real project should still stay in normal Python files.

## 1. Notebook Purpose

This section explains what we are doing before we run any code.

The goal of this notebook is to help you:
- summarize the project
- explain the project structure
- simulate a simple credit scoring workflow
- note strengths and weaknesses
- list possible new features
- prepare questions for your guide

Because this notebook is standalone, it uses a **toy scoring model** rather than your actual trained model.

In [10]:
# This cell imports only Python standard-library modules.
# These are available in almost every Python installation,
# which is why this notebook stays independent.

from pathlib import Path
import json
import math
import random
import statistics
from textwrap import fill

# We set a random seed so results are reproducible.
# Reproducible means the notebook gives the same random values
# each time you run it, which is helpful for demos.
random.seed(42)

print('Standalone notebook loaded successfully.')
print('Only standard-library modules are being used.')


Standalone notebook loaded successfully.
Only standard-library modules are being used.


## 2. Basic Project Summary

This cell stores a clean summary of your project in plain Python data.

This is useful because:
- it makes your explanation structured
- it can be reused later in the notebook
- it is easier to update than rewriting the same explanation in many places

In [11]:
# This dictionary is a simple human-readable summary of your project.
# You can edit any of these lines if you want the notebook to reflect
# a slightly different version of your final narrative.

project_summary = {
    'project_name': 'MasterMind AI Credit Scoring',
    'core_goal': 'Predict probability of default and support explainable credit decisions.',
    'main_parts': [
        'Data preparation and cleaning',
        'Feature engineering',
        'Model training and calibration',
        'Explainability and fairness review',
        'Flask API and user interface'
    ],
    'decision_labels': ['APPROVE', 'REVIEW', 'DECLINE'],
    'special_strengths': [
        'Modular project structure',
        'Explainability-aware design',
        'Fairness audit idea',
        'API plus frontend presentation layer'
    ]
}

print(json.dumps(project_summary, indent=2))


{
  "project_name": "MasterMind AI Credit Scoring",
  "core_goal": "Predict probability of default and support explainable credit decisions.",
  "main_parts": [
    "Data preparation and cleaning",
    "Feature engineering",
    "Model training and calibration",
    "Explainability and fairness review",
    "Flask API and user interface"
  ],
  "decision_labels": [
    "APPROVE",
    "REVIEW",
    "DECLINE"
  ],
  "special_strengths": [
    "Modular project structure",
    "Explainability-aware design",
    "Fairness audit idea",
    "API plus frontend presentation layer"
  ]
}


## 3. Project Modules Overview

This cell gives a simple table-like overview.

It does not inspect your actual codebase. Instead, it gives you a clear presentation-friendly structure you can show while explaining your project.

In [12]:
# Each tuple represents one conceptual module of the project.
# This is useful for presentations and viva-style discussions.

modules = [
    ('Module 1', 'Data Pipeline', 'Load raw data, clean traps, split data, and create processed outputs.'),
    ('Module 2', 'Feature Engineering', 'Create application-level and aggregate features for modeling.'),
    ('Module 3', 'Model Training', 'Train the credit-risk model and calibrate default probabilities.'),
    ('Module 4', 'Explainability and Fairness', 'Generate model explanations and audit fairness behavior.'),
    ('Module 5', 'API and UI', 'Serve predictions through a Flask backend and frontend interface.')
]

for code, title, description in modules:
    print(f'{code:10} | {title:28} | {description}')


Module 1   | Data Pipeline                | Load raw data, clean traps, split data, and create processed outputs.
Module 2   | Feature Engineering          | Create application-level and aggregate features for modeling.
Module 3   | Model Training               | Train the credit-risk model and calibrate default probabilities.
Module 4   | Explainability and Fairness  | Generate model explanations and audit fairness behavior.
Module 5   | API and UI                   | Serve predictions through a Flask backend and frontend interface.


## 4. Simple Standalone Credit Scoring Demo

This is the most important standalone part of the notebook.

Because we are not importing your real trained model, we create a **toy scoring function** that imitates the idea of credit scoring.

This function is not meant to be production-grade. It is meant to help you explain:
- what input fields matter
- how risk score can be built from features
- how a probability can be derived
- how decisions like APPROVE / REVIEW / DECLINE are made

In [13]:
# This helper safely divides one value by another.
# If the denominator is zero, we return 0.0 instead of crashing.
# This mirrors a common pattern in feature engineering.
def safe_div(a, b):
    if b in (0, 0.0, None):
        return 0.0
    return a / b


# This helper converts a raw risk score into a 0-1 probability.
# The sigmoid function is commonly used in machine learning
# to turn a score into a probability-like output.
def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))


# This is a small, readable scoring function.
# It uses a few understandable rules:
# - higher credit relative to income increases risk
# - higher annuity relative to income increases risk
# - lower external source scores increase risk
# - more social defaults / risk history increase risk
#
# The output is a dictionary with:
# - engineered features
# - probability_of_default
# - decision label
# - top drivers
def demo_credit_score(application):
    income = float(application.get('AMT_INCOME_TOTAL_CAPPED', 0.0))
    credit = float(application.get('AMT_CREDIT', 0.0))
    annuity = float(application.get('AMT_ANNUITY', 0.0))
    ext1 = float(application.get('EXT_SOURCE_1', 0.0))
    ext2 = float(application.get('EXT_SOURCE_2', 0.0))
    ext3 = float(application.get('EXT_SOURCE_3', 0.0))
    obs30 = float(application.get('OBS_30_CNT_SOCIAL_CIRCLE', 0.0))
    def30 = float(application.get('DEF_30_CNT_SOCIAL_CIRCLE', 0.0))
    obs60 = float(application.get('OBS_60_CNT_SOCIAL_CIRCLE', 0.0))
    def60 = float(application.get('DEF_60_CNT_SOCIAL_CIRCLE', 0.0))

    # These engineered features are simple versions of real risk features.
    credit_income_ratio = safe_div(credit, income)
    annuity_income_ratio = safe_div(annuity, income)
    ext_source_mean = (ext1 + ext2 + ext3) / 3.0
    social_risk_sum = obs30 + def30 + obs60 + def60

    # We calculate a toy risk score.
    # Higher is worse.
    # Lower ext_source_mean is riskier, so we subtract it from 1.
    raw_score = (
        1.30 * credit_income_ratio +
        2.20 * annuity_income_ratio +
        1.80 * (1.0 - ext_source_mean) +
        0.08 * social_risk_sum
    )

    # We shift the score slightly to keep output probabilities reasonable.
    probability_of_default = sigmoid(raw_score - 1.75)

    # Policy logic:
    # - below 0.15 -> APPROVE
    # - below 0.35 -> REVIEW
    # - else -> DECLINE
    if probability_of_default < 0.15:
        decision = 'APPROVE'
    elif probability_of_default < 0.35:
        decision = 'REVIEW'
    else:
        decision = 'DECLINE'

    # We create simple explanatory drivers.
    # In your real project this role is played by SHAP/explainability logic.
    drivers = {
        'credit_income_ratio': credit_income_ratio,
        'annuity_income_ratio': annuity_income_ratio,
        'low_external_score_risk': (1.0 - ext_source_mean),
        'social_risk_sum': social_risk_sum
    }

    # Sort drivers by magnitude so the most important ones appear first.
    top_drivers = sorted(drivers.items(), key=lambda item: abs(item[1]), reverse=True)

    return {
        'engineered_features': {
            'credit_income_ratio': round(credit_income_ratio, 4),
            'annuity_income_ratio': round(annuity_income_ratio, 4),
            'ext_source_mean': round(ext_source_mean, 4),
            'social_risk_sum': round(social_risk_sum, 4)
        },
        'probability_of_default': round(probability_of_default, 4),
        'decision': decision,
        'top_drivers': top_drivers
    }


print('Standalone demo scoring function is ready.')


Standalone demo scoring function is ready.


## 5. Create a Sample Applicant

This cell defines one example applicant.

You can change these values later and rerun the scoring cell to see how the output changes.

In [14]:
# This sample payload is intentionally simple and self-contained.
# It mirrors the type of information a credit model might use.

sample_applicant = {
    'AMT_INCOME_TOTAL_CAPPED': 120000.0,
    'AMT_CREDIT': 250000.0,
    'AMT_ANNUITY': 25000.0,
    'EXT_SOURCE_1': 0.20,
    'EXT_SOURCE_2': 0.40,
    'EXT_SOURCE_3': 0.60,
    'OBS_30_CNT_SOCIAL_CIRCLE': 1.0,
    'DEF_30_CNT_SOCIAL_CIRCLE': 0.0,
    'OBS_60_CNT_SOCIAL_CIRCLE': 1.0,
    'DEF_60_CNT_SOCIAL_CIRCLE': 0.0
}

print(json.dumps(sample_applicant, indent=2))


{
  "AMT_INCOME_TOTAL_CAPPED": 120000.0,
  "AMT_CREDIT": 250000.0,
  "AMT_ANNUITY": 25000.0,
  "EXT_SOURCE_1": 0.2,
  "EXT_SOURCE_2": 0.4,
  "EXT_SOURCE_3": 0.6,
  "OBS_30_CNT_SOCIAL_CIRCLE": 1.0,
  "DEF_30_CNT_SOCIAL_CIRCLE": 0.0,
  "OBS_60_CNT_SOCIAL_CIRCLE": 1.0,
  "DEF_60_CNT_SOCIAL_CIRCLE": 0.0
}


## 6. Score the Sample Applicant

This cell runs the standalone demo function and prints the result clearly.

In [15]:
# We run the sample applicant through our demo scoring function.
# The result contains engineered features, probability, decision,
# and a ranked list of drivers.

sample_result = demo_credit_score(sample_applicant)
print(json.dumps(sample_result, indent=2))


{
  "engineered_features": {
    "credit_income_ratio": 2.0833,
    "annuity_income_ratio": 0.2083,
    "ext_source_mean": 0.4,
    "social_risk_sum": 2.0
  },
  "probability_of_default": 0.9344,
  "decision": "DECLINE",
  "top_drivers": [
    [
      "credit_income_ratio",
      2.0833333333333335
    ],
    [
      "social_risk_sum",
      2.0
    ],
    [
      "low_external_score_risk",
      0.5999999999999999
    ],
    [
      "annuity_income_ratio",
      0.20833333333333334
    ]
  ]
}


## 7. Make the Output Easier to Explain

This cell converts raw driver names into business-friendly explanations.

That mirrors the idea of explainability in your main project.

In [16]:
# This mapping acts like a simple explanation dictionary.
# In a real project, these explanations can be linked to SHAP values,
# business rules, or model interpretation logic.

reason_map = {
    'credit_income_ratio': 'Requested credit is high compared with stated income.',
    'annuity_income_ratio': 'Repayment burden is high compared with stated income.',
    'low_external_score_risk': 'External source scores suggest elevated repayment risk.',
    'social_risk_sum': 'Social-circle indicators suggest slightly higher credit risk.'
}

print('Top decision drivers in business language:')
for name, value in sample_result['top_drivers']:
    print('-', name, '->', reason_map.get(name, 'General model risk contribution.'), f'(score={round(value, 4)})')


Top decision drivers in business language:
- credit_income_ratio -> Requested credit is high compared with stated income. (score=2.0833)
- social_risk_sum -> Social-circle indicators suggest slightly higher credit risk. (score=2.0)
- low_external_score_risk -> External source scores suggest elevated repayment risk. (score=0.6)
- annuity_income_ratio -> Repayment burden is high compared with stated income. (score=0.2083)


## 8. Try Multiple Applicants

This cell creates a small synthetic set of applicants.

It helps you explain that a scoring system should behave differently for different profiles.

In [17]:
# This function creates a synthetic applicant profile.
# We use random values in sensible ranges to create demo data.

def random_applicant():
    income = random.uniform(50000, 300000)
    credit = income * random.uniform(0.8, 3.5)
    annuity = credit / random.uniform(8, 25)
    return {
        'AMT_INCOME_TOTAL_CAPPED': round(income, 2),
        'AMT_CREDIT': round(credit, 2),
        'AMT_ANNUITY': round(annuity, 2),
        'EXT_SOURCE_1': round(random.uniform(0.1, 0.9), 2),
        'EXT_SOURCE_2': round(random.uniform(0.1, 0.9), 2),
        'EXT_SOURCE_3': round(random.uniform(0.1, 0.9), 2),
        'OBS_30_CNT_SOCIAL_CIRCLE': float(random.randint(0, 3)),
        'DEF_30_CNT_SOCIAL_CIRCLE': float(random.randint(0, 1)),
        'OBS_60_CNT_SOCIAL_CIRCLE': float(random.randint(0, 3)),
        'DEF_60_CNT_SOCIAL_CIRCLE': float(random.randint(0, 1))
    }


# Generate a small batch of demo applicants.
demo_applicants = [random_applicant() for _ in range(10)]

# Score each applicant and store the result.
demo_results = []
for idx, applicant in enumerate(demo_applicants, start=1):
    result = demo_credit_score(applicant)
    demo_results.append({
        'id': idx,
        'income': applicant['AMT_INCOME_TOTAL_CAPPED'],
        'credit': applicant['AMT_CREDIT'],
        'pd': result['probability_of_default'],
        'decision': result['decision']
    })

for row in demo_results:
    print(row)


{'id': 1, 'income': 209856.7, 'credit': 182056.78, 'pd': 0.6088, 'decision': 'DECLINE'}
{'id': 2, 'income': 73423.81, 'credit': 104862.74, 'pd': 0.8192, 'decision': 'DECLINE'}
{'id': 3, 'income': 252357.61, 'credit': 206314.12, 'pd': 0.665, 'decision': 'DECLINE'}
{'id': 4, 'income': 74179.09, 'credit': 229082.46, 'pd': 0.9768, 'decision': 'DECLINE'}
{'id': 5, 'income': 194338.04, 'credit': 525168.22, 'pd': 0.9849, 'decision': 'DECLINE'}
{'id': 6, 'income': 163352.58, 'credit': 498568.09, 'pd': 0.9762, 'decision': 'DECLINE'}
{'id': 7, 'income': 165565.05, 'credit': 253125.64, 'pd': 0.8154, 'decision': 'DECLINE'}
{'id': 8, 'income': 150291.2, 'credit': 147091.38, 'pd': 0.7465, 'decision': 'DECLINE'}
{'id': 9, 'income': 116220.04, 'credit': 170366.29, 'pd': 0.8205, 'decision': 'DECLINE'}
{'id': 10, 'income': 177381.57, 'credit': 185444.53, 'pd': 0.7889, 'decision': 'DECLINE'}


## 9. Simple Summary Statistics

This cell gives small descriptive summaries from the synthetic demo results.

This is useful because many viva or review discussions ask things like:
- what is the average predicted risk?
- how many applicants were approved?
- how many were sent to review?
- how many were declined?

In [18]:
# Extract probabilities for simple descriptive analysis.
pds = [row['pd'] for row in demo_results]

# Count decisions.
decision_counts = {'APPROVE': 0, 'REVIEW': 0, 'DECLINE': 0}
for row in demo_results:
    decision_counts[row['decision']] += 1

summary_stats = {
    'mean_probability_of_default': round(statistics.mean(pds), 4),
    'min_probability_of_default': round(min(pds), 4),
    'max_probability_of_default': round(max(pds), 4),
    'decision_counts': decision_counts
}

print(json.dumps(summary_stats, indent=2))


{
  "mean_probability_of_default": 0.8202,
  "min_probability_of_default": 0.6088,
  "max_probability_of_default": 0.9849,
  "decision_counts": {
    "APPROVE": 0,
    "REVIEW": 0,
    "DECLINE": 10
  }
}


## 10. Strengths of the Real Project

This is a structured review cell.

It helps you talk about the project professionally instead of only saying 'it works'.

In [19]:
# These strengths are written as concise evaluation points.
# You can reuse them in reports, viva answers, or guide discussions.

strengths = [
    'The project is modular and separated into clear components instead of being one large script.',
    'It goes beyond basic accuracy by including explainability, fairness thinking, and API design.',
    'It has stronger product direction than a typical student ML notebook-only project.',
    'It is easier to maintain because training-time and runtime concerns are conceptually separated.',
    'It is already close to a demo-ready prototype because it includes a backend and UI layer.'
]

for idx, item in enumerate(strengths, start=1):
    print(f'{idx}. {fill(item, width=95)}')


1. The project is modular and separated into clear components instead of being one large script.
2. It goes beyond basic accuracy by including explainability, fairness thinking, and API design.
3. It has stronger product direction than a typical student ML notebook-only project.
4. It is easier to maintain because training-time and runtime concerns are conceptually separated.
5. It is already close to a demo-ready prototype because it includes a backend and UI layer.


## 11. Improvement Areas

This cell focuses on constructive criticism.

That is important because a strong academic/project review should identify:
- what is good
- what is incomplete
- what should be improved before adding more complexity

In [20]:
# These are high-value improvement areas based on the overall project shape.
# They are phrased in a guide-friendly way.

improvements = [
    'Improve consistency between frontend behavior, backend contract, and project documentation.',
    'Strengthen automated testing, especially API-level tests and end-to-end validation.',
    'Tighten artifact status reporting so the fairness and model states are always trustworthy.',
    'Document the exact expected data flow more clearly for future maintenance and presentation.',
    'Reduce mismatch between demo assumptions and production assumptions.'
]

for idx, item in enumerate(improvements, start=1):
    print(f'{idx}. {fill(item, width=95)}')


1. Improve consistency between frontend behavior, backend contract, and project documentation.
2. Strengthen automated testing, especially API-level tests and end-to-end validation.
3. Tighten artifact status reporting so the fairness and model states are always trustworthy.
4. Document the exact expected data flow more clearly for future maintenance and presentation.
5. Reduce mismatch between demo assumptions and production assumptions.


## 12. Best Feature Ideas to Add

This section helps you think about what to add next without making the project unnecessarily complicated.

The goal here is not to add random features.
The goal is to add features that are:
- useful
- realistic
- visible in a demo
- aligned with your current project

In [21]:
# Each feature idea includes a short reason.
# This makes it easier to justify the idea to a guide.

feature_ideas = [
    {
        'feature': 'Batch scoring with CSV upload',
        'why_it_fits': 'It extends the API/UI naturally and makes the project more practical for analyst use.'
    },
    {
        'feature': 'Threshold simulation dashboard',
        'why_it_fits': 'It shows business understanding by letting the user explore APPROVE/REVIEW/DECLINE tradeoffs.'
    },
    {
        'feature': 'Data drift monitoring report',
        'why_it_fits': 'It makes the project feel more production-aware and responsible.'
    },
    {
        'feature': 'What-if applicant analysis',
        'why_it_fits': 'It builds naturally on explainability and makes the project more interactive.'
    },
    {
        'feature': 'Downloadable decision report',
        'why_it_fits': 'It is easy to present and useful for documentation and audit-style workflows.'
    }
]

for idx, idea in enumerate(feature_ideas, start=1):
    print(f"{idx}. {idea['feature']}")
    print('   Reason:', idea['why_it_fits'])


1. Batch scoring with CSV upload
   Reason: It extends the API/UI naturally and makes the project more practical for analyst use.
2. Threshold simulation dashboard
   Reason: It shows business understanding by letting the user explore APPROVE/REVIEW/DECLINE tradeoffs.
3. Data drift monitoring report
   Reason: It makes the project feel more production-aware and responsible.
4. What-if applicant analysis
   Reason: It builds naturally on explainability and makes the project more interactive.
5. Downloadable decision report
   Reason: It is easy to present and useful for documentation and audit-style workflows.


## 13. Which Features Might Be Overkill?

This section is useful because guides often care about scope control.

A project becomes weaker, not stronger, if it tries to do everything and finishes nothing properly.

In [22]:
# These are examples of ideas that may be too large for the current scope.
# This does not mean they are bad. It only means they may not be the best next step.

overkill_features = [
    'Full MLOps deployment pipeline',
    'Continuous online retraining',
    'Complex ensemble/champion-challenger system',
    'Advanced fairness expansion across many protected or proxy dimensions without a clear requirement',
    'Heavy cloud deployment before local reliability and testing are fully polished'
]

for idx, item in enumerate(overkill_features, start=1):
    print(f'{idx}. {fill(item, width=95)}')


1. Full MLOps deployment pipeline
2. Continuous online retraining
3. Complex ensemble/champion-challenger system
4. Advanced fairness expansion across many protected or proxy dimensions without a clear
requirement
5. Heavy cloud deployment before local reliability and testing are fully polished


## 14. Questions to Ask Your Guide

This is one of the most useful sections in the notebook.

These questions are designed to help you check whether a new feature is:
- relevant
- acceptable
- worth the time
- or just overkill

In [23]:
# These questions are written in a practical and respectful way.
# They should help you get concrete feedback from your guide.

guide_questions = [
    'Should I first improve testing, consistency, and documentation before adding a new feature?',
    'For evaluation, should this project be treated mainly as an ML project, a software project, or a product prototype?',
    'Would batch scoring be a meaningful extension for this project, or is it outside the expected scope?',
    'Would a threshold simulation dashboard add useful business insight, or is it unnecessary here?',
    'Would drift monitoring be considered a strong improvement, or would it be too advanced for this stage?',
    'Would a what-if analysis feature be relevant and useful, or would it be overkill?',
    'Should I focus more on stronger validation and testing, or on one additional visible feature?',
    'Which final upgrade would you recommend most: batch scoring, threshold simulation, drift monitoring, or what-if analysis?'
]

for idx, question in enumerate(guide_questions, start=1):
    print(f'{idx}. {fill(question, width=95)}')


1. Should I first improve testing, consistency, and documentation before adding a new feature?
2. For evaluation, should this project be treated mainly as an ML project, a software project, or
a product prototype?
3. Would batch scoring be a meaningful extension for this project, or is it outside the expected
scope?
4. Would a threshold simulation dashboard add useful business insight, or is it unnecessary here?
5. Would drift monitoring be considered a strong improvement, or would it be too advanced for this
stage?
6. Would a what-if analysis feature be relevant and useful, or would it be overkill?
7. Should I focus more on stronger validation and testing, or on one additional visible feature?
8. Which final upgrade would you recommend most: batch scoring, threshold simulation, drift
monitoring, or what-if analysis?


## 15. Recommended Final Direction

This cell gives a concise recommendation you can say confidently.

A good final direction for your project is usually:
1. tighten quality and consistency
2. add one strong visible feature
3. avoid adding too many advanced extras

In [24]:
# This recommendation is intentionally short and presentation-friendly.

recommended_plan = {
    'first_priority': 'Improve project consistency, testing, and documentation quality.',
    'best_next_feature': 'Batch scoring with CSV upload or threshold simulation dashboard.',
    'features_to_avoid_for_now': 'Large MLOps or over-engineered research extensions.',
    'final_message': 'Make the current system stronger first, then add one visible, practical feature.'
}

print(json.dumps(recommended_plan, indent=2))


{
  "first_priority": "Improve project consistency, testing, and documentation quality.",
  "best_next_feature": "Batch scoring with CSV upload or threshold simulation dashboard.",
  "features_to_avoid_for_now": "Large MLOps or over-engineered research extensions.",
  "final_message": "Make the current system stronger first, then add one visible, practical feature."
}


## 16. Personal Notes Cell

Use this last cell to write your own observations after discussing with your guide.

Because it is just a normal Python list, you can edit it however you want.

In [25]:
# Add your own notes below.
# Example uses:
# - what your guide preferred
# - which feature was approved
# - what to fix first
# - what to cut from the scope

my_notes = [
    'Guide feedback goes here.',
    'Approved feature goes here.',
    'Immediate fixes go here.'
]

for idx, note in enumerate(my_notes, start=1):
    print(f'{idx}. {note}')


1. Guide feedback goes here.
2. Approved feature goes here.
3. Immediate fixes go here.
